# AI Song Recommender Chatbot

This notebook demonstrates the core functionality of the AI Song Recommender Chatbot, including loading the massive Spotify dataset, filtering songs by audio features (valence and energy), and using a deep learning Hugging Face Zero-Shot Classifier to detect user intents from natural language.


## 1. Imports and Setup

In [ ]:
import pandas as pd
from transformers import pipeline
import random

## 2. Load the Dataset
We load the massive Spotify features dataset containing over 200,000 tracks and filter for popularity.

In [ ]:
df = pd.read_csv('data/SpotifyFeatures.csv')
# Clean duplicates and obscure tracks
df = df.drop_duplicates(subset=['track_name', 'artist_name'])
df = df[df['popularity'] >= 40]
print(f"Loaded {len(df)} popular tracks.")
df.head()

## 3. Initialize the Deep Learning NLP Model
We use a pre-trained zero-shot classification model to infer the user's mood or desired genre from raw text.

In [ ]:
print("Loading classifier...")
classifier = pipeline("zero-shot-classification", model="cross-encoder/nli-distilroberta-base")
candidate_labels = ["happy", "sad", "relaxed", "motivated", "pop", "rock", "hip-hop", "jazz", "classical", "world"]

user_input = "I am feeling really exhausted and just want to chill."
result = classifier(user_input, candidate_labels)
best_intent = result['labels'][0]
print(f"User Input: '{user_input}'\nDetected Intent: {best_intent.upper()}")

## 4. Recommendation Engine
Now we filter the dataset based on the detected intent (mapping mood to valence/energy).

In [ ]:
def get_recommendations(intent, num_tracks=3):
    filtered_df = df.copy()
    if intent == "happy":
        filtered_df = filtered_df[(filtered_df['valence'] >= 0.6) & (filtered_df['energy'] >= 0.6)]
    elif intent == "sad":
        filtered_df = filtered_df[(filtered_df['valence'] <= 0.4) & (filtered_df['energy'] <= 0.5)]
    elif intent == "relaxed":
        filtered_df = filtered_df[(filtered_df['energy'] <= 0.5) & (filtered_df['valence'] > 0.3)]
    elif intent == "motivated":
        filtered_df = filtered_df[filtered_df['energy'] >= 0.75]
    else:
        filtered_df = filtered_df[filtered_df['genre'].str.lower() == intent]
        
    return filtered_df.sample(num_tracks)[['track_name', 'artist_name', 'genre']]

print("Recommendations for", best_intent)
get_recommendations(best_intent)